In [2]:
import pandas as pd 
import numpy as np

In [3]:
import os

from io import BytesIO
from datetime import datetime, timezone
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [4]:
def read_parquet_prefix(s3_client, bucket_name, prefix):
    """
    Read every Parquet object under a MinIO prefix
    and return one pandas DataFrame.
    """
    #MinIO may contain many objects. Instead of returning all objects in one response, the S3 API may divide them into pages.
    paginator = s3_client.get_paginator("list_objects_v2")

    #This list will store the names of all Parquet objects found.
    parquet_keys = []

    #This asks MinIO: Show me all objects inside this bucket whose key starts with this prefix.
    for page in paginator.paginate(Bucket=bucket_name,Prefix=prefix):
        #Contents contains the objects found by MinIO.
        for obj in page.get("Contents", []):
            #The key is the complete MinIO path of the file.
            key = obj["Key"]

            if key.endswith(".parquet"):
                parquet_keys.append(key)

    if not parquet_keys:
        raise FileNotFoundError(
            f"No Parquet files found under prefix: {prefix}"
        )

    dataframes = []

    for key in parquet_keys:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key
        )

        parquet_bytes = response["Body"].read()

        dataframe = pd.read_parquet(
            BytesIO(parquet_bytes),
            engine="fastparquet"
        )

        dataframes.append(dataframe)

        print(f"Loaded: {key}")
        print(f"Shape: {dataframe.shape}")

    combined_dataframe = pd.concat(
        dataframes,
        ignore_index=True
    )

    return combined_dataframe

def read_parquet_object(s3_client, bucket_name, object_key):
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    return pd.read_parquet(
        BytesIO(response["Body"].read()),
        engine="fastparquet",
    )

In [5]:
metadata_df = read_parquet_prefix(
    s3_client,
    BUCKET_NAME,
    "metadata/etl_metadata/"
)

required_silver_datasets = {"train_data", "test_data", "RUL_data"}
successful_silver_metadata = metadata_df[
    (metadata_df["layer"] == "silver")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_silver_datasets)
]

datasets_by_run = successful_silver_metadata.groupby("run_id")["dataset_name"].apply(set)
valid_silver_run_ids = sorted(
    run_id
    for run_id, datasets in datasets_by_run.items()
    if required_silver_datasets.issubset(datasets)
)

if not valid_silver_run_ids:
    raise FileNotFoundError("No successful Silver run contains train, test, and RUL data.")

pipeline_run_id = valid_silver_run_ids[-1]
selected_silver_metadata = successful_silver_metadata[
    successful_silver_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_silver_metadata["load_date"].iloc[0]
metadata_key = (
    f"metadata/etl_metadata/load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/manifest.parquet"
)

def silver_object_key(dataset_name):
    matches = selected_silver_metadata[
        selected_silver_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Silver object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

silver_train_df = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("train_data")
)
silver_test_df = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("test_data")
)
silver_rul_df = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("RUL_data")
)

print(f"Selected Silver run: {pipeline_run_id}")

Loaded: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
Shape: (7, 8)
Selected Silver run: 20260902T120041719003Z


In [6]:
rul_training_features = silver_train_df.copy()

In [7]:
temp1 =  rul_training_features.groupby("unit_number").agg(last_cycle=("time_in_cycles","max"))

In [8]:
rul_training_features = rul_training_features.merge(temp1,on="unit_number",how="left")

In [9]:
rul_training_features["remaining_useful_life"] = rul_training_features["last_cycle"]-rul_training_features["time_in_cycles"]

In [10]:
rul_training_features = rul_training_features.drop(columns=["last_cycle","processing_timestamp","data_quality_status"])

In [11]:
rul_training_features.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_13,sensor_measurement_14,sensor_measurement_15,sensor_measurement_16,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21,remaining_useful_life
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393.0,2388.0,100.0,38.90,23.4044,187


In [ ]:
parquet_buffer = BytesIO()

rul_training_features.to_parquet(
    parquet_buffer,
    index=False,
    engine="fastparquet",
)

parquet_buffer.seek(0)
object_key = (
    f"gold/rul_training_features/"
    f"load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/"
    f"rul.parquet"
)

s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=object_key,
    Body=parquet_buffer.getvalue(),
    ContentType="application/octet-stream",
)

print(f"Uploaded Gold dataset to MinIO: {object_key}")

etl_metadata_df = read_parquet_object(
    s3_client, BUCKET_NAME, metadata_key
)
etl_metadata_df = etl_metadata_df[
    ~((etl_metadata_df["run_id"] == pipeline_run_id)
      & (etl_metadata_df["layer"] == "gold")
      & (etl_metadata_df["dataset_name"] == "rul_training_features"))
]
etl_metadata_df = pd.concat(
    [
        etl_metadata_df,
        pd.DataFrame([{
            "run_id": pipeline_run_id,
            "load_date": pipeline_load_date,
            "layer": "gold",
            "dataset_name": "rul_training_features",
            "object_key": object_key,
            "status": "success",
            "row_count": len(rul_training_features),
            "recorded_at": datetime.now(timezone.utc).isoformat(),
        }]),
    ],
    ignore_index=True,
)
metadata_buffer = BytesIO()
etl_metadata_df.to_parquet(
    metadata_buffer,
    index=False,
    engine="fastparquet",
)
metadata_buffer.seek(0)
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=metadata_key,
    Body=metadata_buffer.getvalue(),
    ContentType="application/octet-stream",
)
print(f"Updated ETL metadata manifest: {metadata_key}")

Uploaded Gold dataset to MinIO: gold/rul_training_features/load_date=2026-09-02/run_id=20260902T120041719003Z/rul.parquet
Updated ETL metadata manifest: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet


## Gold-layer documentation

### `gold/rul_training_features/`

**Purpose:** Training-ready data for remaining-useful-life prediction.

**Grain:** One row represents one engine at one operating cycle.

**Source:** The Silver training dataset read from MinIO.

**Target:** `remaining_useful_life`, calculated as the engine's final observed training cycle minus its current cycle.

The original standardized feature columns are preserved. No sensor was removed at this stage. Feature engineering and feature-selection decisions will be handled during exploratory analysis and modeling.

### Planned Gold datasets

- `gold/current_engine_health/`: one row per engine at its latest observed cycle. This will be created after health indicators and current-condition logic are defined.
- `gold/maintenance_predictions/`: one row per engine prediction event. This requires a trained model and prediction timestamp.
- `gold/model_performance/`: one row per metric from one model evaluation run. This requires the modeling and evaluation workflow.

These datasets are intentionally not populated yet because valid records cannot be created before the corresponding analysis and modeling steps are complete.

### Object-key note

The current object key uses the short filename `rul.parquet` to avoid the previous key-length error. Re-running on the same date will replace the object at that exact key. Preserving multiple Gold runs can be added later with a run identifier or timestamp in the filename.